## Import Libraries

In [2]:
import os
from os.path import exists
import glob
import json
import pdb
import numpy as np
from skimage import draw
import matplotlib.pyplot as plt
import argparse
import pyfiglet
from skimage import measure
from tqdm import tqdm
from PIL import Image
import pyvips as Vips
# import openslide
from Reinhard import Reinhard
import pandas as pd
import random
import shutil
import albumentations as A

## Generate Crops

In [3]:
ID_MASK_SHAPE = (1024, 1024)

# Color Coding
lablel2id = {'True':'50', 'Pre':'100',
             'False':'150', 'Unknown':'0'}

#DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/"
#DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/generated_data_v2/Unnormalized/"

In [4]:
DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/patient_wise_crops_LB_v2/"

In [10]:

def normalization(REF_IMG_PATH):
    print("Init Normalization")
    ref_image = Vips.Image.new_from_file(REF_IMG_PATH)
    normalizer = Reinhard()
    normalizer.fit(ref_image)
    return normalizer


def save_img(img, file_name, tileX, tileY, save_dir, label="mask"):
    im = Image.fromarray(img)

    file_name = file_name + "_" + str(tileX)+"x" + "_" + str(tileY) + "y" + "_" + label + ".png"

    save_name = os.path.join(save_dir, file_name)
    im.save(save_name)


def polygon2id(image_shape, mask, ids, coords_x, coords_y):
    vertex_row_coords, vertex_col_coords = coords_y, coords_x
    fill_row_coords, fill_col_coords = draw.polygon(
        vertex_row_coords, vertex_col_coords, image_shape)

    # Row and col are flipped
    mask[fill_col_coords, fill_row_coords] = ids
    return mask

def polygon2mask1(image_shape, mask, color, coords_x, coords_y):
    """Compute a mask with labels having different colors
    from polygon.
    Parameters
    ----------
    image_shape : tuple of size 2.
        The shape of the mask.
    coords_x: X coordinates
    coords_y: Y coordinates
    mask : Mask with same size of the image (initially empty
    mask is given as input)
    Returns
    -------
    mask : 2-D ndarray of type 'bool'.
        The mask that corresponds to the input polygon.
    """

    vertex_row_coords, vertex_col_coords = coords_x, coords_y
    fill_row_coords, fill_col_coords = draw.polygon(vertex_row_coords, vertex_col_coords, image_shape)

    # Row and col are flipped
    mask[fill_col_coords, fill_row_coords] = color

    # mask[fill_row_coords, fill_col_coords] = color
    return mask

In [11]:
def get_vips_info(vips_img):
    # # Get bounds-x and bounds-y offeset
    #print(vips_img.get_fields())
    vfields = [f.split('.') for f in vips_img.get_fields()]
    #print("--------------",vfields)
    #vfields = [f for f in vfields if f[0] == 'openslide']
    vfields = [f for f in vfields]
    vfields = dict([('.'.join(k[1:]), vips_img.get('.'.join(k))) for k in vfields])
    print(vfields)
    return vfields

In [12]:
def process_json(WSI_path, json_path,  visualize=False):
    """This function is used to read and process the json files
    and generate save generated masks

    Parameters
    -----------
    json_path : path to json file
    save_dir : dir where the generated masks will be saved
    visualize : True , if you want to see the mask generated
    """


    # Mask Folder
    mask_save_dir = os.path.join(DATASET_PATH, "labels")
    if not os.path.exists(mask_save_dir):
        os.makedirs(mask_save_dir)

    # Image Folder
    image_save_dir = os.path.join(DATASET_PATH, "images")
    if not os.path.exists(image_save_dir):
        os.makedirs(image_save_dir)


    imagenames = glob.glob(os.path.join(WSI_path, "*.svs"))
    imagenames = sorted(imagenames)
    
    plaque_dict = {'True': 0, 'Pre': 0, 'False': 0,'Unknown': 0}

    for img in imagenames:
        # Read the WSI image
        vips_img = Vips.Image.new_from_file(img, level=0)
        vinfo = get_vips_info(vips_img)
        # Get the corresponding json file
        # json_file_name = os.path.basename(img).split(".svs")[0] + ".json"
        json_file_name = os.path.basename(img) + ".json"
        json_file_name = os.path.join(json_path, json_file_name)
        # json_file_list = [json_file_name, "/home/vivek/Datasets/AmyB/amyb_wsi/XE19-010_1_AmyB_1_1.json"]
        # merge_json(json_file_list, "/home/vivek/Datasets/AmyB/amyb_wsi/test.json")
        # json_file_name = os.path.join(os.path.dirname(img), "XE19-010_1_AmyB_1_37894x_177901y_image.png[--series, 0].json")

        # json_file_name = "/home/vivek/Datasets/AmyB/amyb_wsi/test.json"
        
        print(json_file_name)
       
        if not exists(json_file_name):
            print("True")
            continue
        
        print("file name : ", json_file_name)

        with open(json_file_name) as f:
            data = json.load(f)
        
        for ele in tqdm(data):

            # Reset ids for each annotation
            ids = 1

            # Create an Empty mask of size similar to image
            id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

            region_id = 0
            prev_label = ""
            i = 0
            plaque_dict[ele['label']] = plaque_dict[ele['label']] + len(ele['region_attributes'])

            for region in ele['region_attributes']:

                # Get tileX and tileY
                tileX = region['tiles'][0]['tileId'][0]
                tileY = region['tiles'][0]['tileId'][1]
                tileWidth = region['tiles'][0]['tileBounds']["WH"][0]
                tileHeight = region['tiles'][0]['tileBounds']["WH"][1]

                # crop the image
                # get the bound-x and bounds-y, offset as Vips crops the empty spaces. Qupath does not
                tileX = (tileX * tileWidth)
                tileY = (tileY * tileHeight)

                vips_img_crop = vips_img.crop(tileX, tileY,tileWidth, tileHeight)
                print(tileX, tileY, tileWidth, tileHeight)
                # Region Bounds
                regX = region["roiBounds"]["XY"][0]
                regY = region["roiBounds"]["XY"][1]
                regWidth = region["roiBounds"]["WH"][0]
                regHeight = region["roiBounds"]["WH"][1]

                # region_crop = vips_img.crop(regX, regY, tileWidth, tileHeight)
                vips_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                                    shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]
                # region_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                #                 shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]

                # unpack from [x,y] to [x], [y]
                coords_x, coords_y = zip(*region['points'])

                coords_x = np.array(coords_x)
                coords_y = np.array(coords_y)

                x1 = tileX
                x2 = tileX + tileWidth 
                y1 = tileY 
                y2 = tileY + tileHeight


                # Remove overlap annotations
                if len(coords_x[coords_x > x2]) > 0 or len(coords_y[coords_y > y2]) > 0:
                    print('Overlap')
                    continue


                # Translate the coordinates to fit within the image crop
                coords_x = np.mod(coords_x, tileWidth)
                coords_y = np.mod(coords_y, tileHeight)



                # label
                label = ele['label']

                if i == 0:
                    ids = int(lablel2id[label])
                elif label == prev_label:
                    ids = int(lablel2id[label])

                # Use polygon2id function to create a mask
                id_mask = polygon2id(ID_MASK_SHAPE, id_mask, ids, coords_y, coords_x)

                # ids = ids + 5

                prev_label = label

                i+=1

                save_img(vips_img_crop, ele['filename'], tileX, tileY, image_save_dir, "image")
                save_img(id_mask, ele['filename'], tileX, tileY, mask_save_dir,"mask")
                id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

            if visualize:
                plt.imshow(id_mask)
                plt.show()

        print(plaque_dict)


def merge_json(json_files, json_output_file=None):
    """
    merge_json: a method to return the combined json of a list of json files

    """
    result = list()
    for f1 in json_files:
        with open(f1, 'r') as infile:
            result.extend(json.load(infile))

    with open(json_output_file, 'w') as output_file:
        json.dump(result, output_file)


In [10]:
dlb_wsi_dir1 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases"
dlb_wsi_dir2 = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/DLB_cases/DLB_cases"
pdd_wsi_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases"
wsi_home_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/"

In [11]:
json_path =  "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/LBD_Jsons_All_Intensity/"

In [12]:
f_list = os.listdir(json_path)
#f_list.remove('.DS_Store')

In [13]:
len(f_list)

53

In [14]:
import os
f_list_updated_names = []
for file in f_list:
    x = file.replace("SYN1", "Syn1")
    os.rename(os.path.join(json_path,file),os.path.join(json_path, x))
    f_list_updated_names.append(x)

In [21]:
f_list_updated_names

['PD130_Syn1_CG.json',
 'PD079_Syn1_CG.json',
 'PD092_Syn1_FCX.json',
 'PD309_Syn1_CG.json',
 'PD306_Syn1_CG.json',
 'PD113_Syn1_CG.json',
 '13_131_CG_aSyn_x200.json',
 'PD271_Syn1_CG.json',
 '14_087_CG_aSyn_x200.json',
 '14_148_CG_aSyn_x200.json',
 'PD110_Syn1_CG.json',
 'PD034_Syn1_CG.json',
 '02_019_Syn1_CG_200x.json',
 'PD002_Syn1_CG.json',
 '15_005_CG_aSyn_x200.json',
 'PD067_Syn1_CG.json',
 '14_133_CG_aSyn_x200.json',
 'PD334_Syn1_CG.json',
 'PD311_Syn1_CG.json',
 '01_104_Syn1_CG_200x.json',
 '17_010_CG_aSyn_x200b.json',
 'PD088_Syn1_CG.json',
 '14_075_CG_aSyn_x200.json',
 '12_060_CG_aSyn_x200.json',
 '01_156_Syn1_CG_200x.json',
 '14_053_CG_aSyn_x200.json',
 'PD041_Syn1_CG.json',
 'PD295_Syn1_CG.json',
 '12_007_CG_aSyn_x200.json',
 '14_153_CG_aSyn_x200.json',
 '00_1108_Syn1_CG_200x.json',
 'PD090_Syn1_CG.json',
 '16_380_CG_aSyn_x200.json',
 '19_046_CG_aSyn_x200.json',
 '21_070_CG_aSyn_x200.json',
 'PD133_Syn1_CG.json',
 '13_177_CG_aSyn_x200.json',
 'PD336_Syn1_CG.json',
 '00_1027

In [22]:
#imagenames = sorted(glob.glob(os.path.join(wsi_home_dir, './*/*.svs')))
dlb_imagenames = sorted(glob.glob(os.path.join(dlb_wsi_dir1, './*.svs')))
pdd_imagenames = sorted(glob.glob(os.path.join(pdd_wsi_dir, '././*.svs')))

In [35]:
rand_image = random.choice(pdd_imagenames)

In [36]:
rand_image

'/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD_Dataset/LBD/PDD_cases/PDD_cases/././PD430_Syn1_CG.svs'

In [32]:
if os.path.join(pdd_wsi_dir,'PD079_Syn1_CG.svs') in pdd_imagenames:
    print(True)

In [15]:
img = os.path.join(pdd_wsi_dir,'PD079_Syn1_CG.svs')

In [37]:
#REF_IMG_PATH = dlb_imagenames[0]  ## v1 is updated using this file
REF_IMG_PATH = rand_image  # Using this image for v2
normalizer = normalization(REF_IMG_PATH)

Init Normalization


In [18]:
"""
mask_save_dir = os.path.join(DATASET_PATH, "labels")
if not os.path.exists(mask_save_dir):
    os.makedirs(mask_save_dir)

# Image Folder
image_save_dir = os.path.join(DATASET_PATH, "images")
if not os.path.exists(image_save_dir):
    os.makedirs(image_save_dir)
"""

In [38]:
count_df = pd.DataFrame(columns=["filename","label","annotation_count"])
filenames = []
labels=[]
len_ann = []
for geofile in f_list_updated_names:
    filename = geofile.replace(".json",".svs")
    geofile1 = os.path.join(json_path,geofile)
    with open(geofile1) as f:
        data = json.load(f)
    for ele in tqdm(data):
        filenames.append(ele["filename"])
        if len(ele['label'])>1:
            labels.append(ele['label'][0]+" "+ ele['label'][1])
        else:
            labels.append(ele['label'][0])
        len_ann.append(len(ele['region_attributes']))
count_df = pd.DataFrame({"filename":filenames,"label":labels,"annotation_count":len_ann})

100%|██████████| 4/4 [00:00<00:00, 17385.72it/s]
0it [00:00, ?it/s]
100%|██████████| 14/14 [00:00<00:00, 187604.65it/s]


In [39]:
count_df[count_df["label"]=="True 3+"]["annotation_count"].sum()

1279

In [40]:
count_df[count_df["label"]=="Pre 3+"]["annotation_count"].sum()

419

In [41]:
count_df[count_df["label"]=="False 3+"]["annotation_count"].sum()

189

In [42]:
count_df[count_df["label"]=="True 2+"]["annotation_count"].sum()

587

In [43]:
count_df[count_df["label"]=="Pre 2+"]["annotation_count"].sum()

75

In [46]:
count_df["filename"].unique()

array(['PD130_Syn1_CG', 'PD079_SYN1_CG', 'PD092_SYN1_FCX',
       'PD309_Syn1_CG', 'PD306_Syn1_CG', 'PD113_Syn1_CG',
       '13_131_CG_aSyn_x200', 'PD271_Syn1_CG', '14_087_CG_aSyn_x200',
       '14_148_CG_aSyn_x200', 'PD110_SYN1_CG', 'PD034_SYN1_CG',
       '02_019_Syn1_CG_200x', 'PD002_SYN1_CG', '15_005_CG_aSyn_x200',
       'PD067_Syn1_CG', '14_133_CG_aSyn_x200', 'PD334_Syn1_CG',
       'PD311_Syn1_CG', '01_104_Syn1_CG_200x', '17_010_CG_aSyn_x200b',
       'PD088_SYN1_CG', '14_075_CG_aSyn_x200', '12_060_CG_aSyn_x200',
       '01_156_Syn1_CG_200x', '14_053_CG_aSyn_x200', 'PD041_SYN1_CG',
       'PD295_Syn1_CG', '12_007_CG_aSyn_x200', '14_153_CG_aSyn_x200',
       '00_1108_Syn1_CG_200x', 'PD090_Syn1_CG', '16_380_CG_aSyn_x200',
       '19_046_CG_aSyn_x200', '21_070_CG_aSyn_x200', 'PD133_Syn1_CG',
       '13_177_CG_aSyn_x200', 'PD336_Syn1_CG', '00_1027_Syn1_CG_200x',
       '11_063_CG_aSyn_x200', 'PD061_Syn1_CG', 'PD149_Syn1_CG',
       '14_036_CG_aSyn_x200', 'PD013_Syn1_CG', 'PD001_Syn1

In [47]:
visualize = False
for geofile in f_list_updated_names:
    filename = geofile.replace(".json",".svs")
    print("**** Processing **** ", filename)
    geofile1 = os.path.join(json_path,geofile)
    with open(geofile1) as f:
        data = json.load(f)
    
    if filename.startswith("PD"):
        continue
        img = os.path.join(pdd_wsi_dir, filename)
    else:
        #try:
         #   img = os.path.join(dlb_wsi_dir1, filename)
        #except:
        img = os.path.join(dlb_wsi_dir2, filename)


    try:
        vips_img = Vips.Image.new_from_file(img, level=0)
    except:
        print("Error loading Vips Image", filename)
        continue
    
    mask_save_dir = os.path.join(DATASET_PATH,filename,"labels")
    if not os.path.exists(mask_save_dir):
        os.makedirs(mask_save_dir)
    else:
        continue

    # Image Folder
    image_save_dir = os.path.join(DATASET_PATH,filename, "images")
    if not os.path.exists(image_save_dir):
        os.makedirs(image_save_dir)

    vips_img = normalizer.transform(vips_img)
    vinfo = get_vips_info(vips_img)
    for ele in tqdm(data):
        # Reset ids for each annotation
        ids = 1

        # Create an Empty mask of size similar to image
        id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

        region_id = 0
        prev_label = ""
        i = 0
        #plaque_dict[ele['label'][0]] = plaque_dict[ele['label'][0]] + len(ele['region_attributes'])
        if len(ele['label'])==1:
            continue
        if ele['label'][1]=='3+' and ele['label'][0] in ["True","Pre"]:
            for region in ele['region_attributes']:

                # Get tileX and tileY
                tileX = region['tiles'][0]['tileId'][0]
                tileY = region['tiles'][0]['tileId'][1]
                tileWidth = region['tiles'][0]['tileBounds']["WH"][0]
                tileHeight = region['tiles'][0]['tileBounds']["WH"][1]

                # crop the image
                # get the bound-x and bounds-y, offset as Vips crops the empty spaces. Qupath does not
                tileX = (tileX * tileWidth)
                tileY = (tileY * tileHeight)

                vips_img_crop = vips_img.crop(tileX, tileY,tileWidth, tileHeight)
                #print(tileX, tileY, tileWidth, tileHeight)
                # Region Bounds
                regX = region["roiBounds"]["XY"][0]
                regY = region["roiBounds"]["XY"][1]
                regWidth = region["roiBounds"]["WH"][0]
                regHeight = region["roiBounds"]["WH"][1]

                # region_crop = vips_img.crop(regX, regY, tileWidth, tileHeight)
                vips_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                                    shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]
                # region_img_crop = np.ndarray(buffer=vips_img_crop.write_to_memory(), dtype=np.uint8,
                #                 shape=(vips_img_crop.height, vips_img_crop.width, vips_img_crop.bands))[..., :3]

                # unpack from [x,y] to [x], [y]
                coords_x, coords_y = zip(*region['points'])

                coords_x = np.array(coords_x)
                coords_y = np.array(coords_y)

                x1 = tileX
                x2 = tileX + tileWidth 
                y1 = tileY 
                y2 = tileY + tileHeight


                # Remove overlap annotations
                if len(coords_x[coords_x > x2]) > 0 or len(coords_y[coords_y > y2]) > 0:
                    print('Overlap')
                    continue


                # Translate the coordinates to fit within the image crop
                coords_x = np.mod(coords_x, tileWidth)
                coords_y = np.mod(coords_y, tileHeight)



                # label
                label = ele['label'][0]

                if i == 0:
                    ids = int(lablel2id[label])
                elif label == prev_label:
                    ids = int(lablel2id[label])

                # Use polygon2id function to create a mask
                id_mask = polygon2id(ID_MASK_SHAPE, id_mask, ids, coords_y, coords_x)

                # ids = ids + 5

                prev_label = label

                i+=1

                save_img(vips_img_crop, ele['filename'], tileX, tileY, image_save_dir, "image")
                save_img(id_mask, ele['filename'], tileX, tileY, mask_save_dir,"mask")
                id_mask = np.zeros(ID_MASK_SHAPE, dtype=np.uint8)

        if visualize:
            plt.imshow(id_mask)
            plt.show()

        #print(plaque_dict)

**** Processing ****  PD130_Syn1_CG.svs
**** Processing ****  PD079_Syn1_CG.svs
**** Processing ****  PD092_Syn1_FCX.svs
**** Processing ****  PD309_Syn1_CG.svs
**** Processing ****  PD306_Syn1_CG.svs
**** Processing ****  PD113_Syn1_CG.svs
**** Processing ****  13_131_CG_aSyn_x200.svs
Error loading Vips Image 13_131_CG_aSyn_x200.svs
**** Processing ****  PD271_Syn1_CG.svs
**** Processing ****  14_087_CG_aSyn_x200.svs
Error loading Vips Image 14_087_CG_aSyn_x200.svs
**** Processing ****  14_148_CG_aSyn_x200.svs
Error loading Vips Image 14_148_CG_aSyn_x200.svs
**** Processing ****  PD110_Syn1_CG.svs
**** Processing ****  PD034_Syn1_CG.svs
**** Processing ****  02_019_Syn1_CG_200x.svs
{'': 'label, macro, thumbnail', 'AppMag': '20', 'DSR ID': '163.1.249.162', 'Date': '01/07/15', 'DisplayColor': '0', 'Exposure Scale': '0.000001', 'Exposure Time': '109', 'Filename': '5697', 'Focus Offset': '-0.000500', 'ICC Profile': 'ScanScope v1', 'ImageID': '5697', 'Left': '16.563103', 'LineAreaXOffset':

100%|██████████| 2/2 [00:01<00:00,  1.03it/s]


**** Processing ****  PD002_Syn1_CG.svs
**** Processing ****  15_005_CG_aSyn_x200.svs
Error loading Vips Image 15_005_CG_aSyn_x200.svs
**** Processing ****  PD067_Syn1_CG.svs
**** Processing ****  14_133_CG_aSyn_x200.svs
Error loading Vips Image 14_133_CG_aSyn_x200.svs
**** Processing ****  PD334_Syn1_CG.svs
**** Processing ****  PD311_Syn1_CG.svs
**** Processing ****  01_104_Syn1_CG_200x.svs
{'': 'label, macro, thumbnail', 'AppMag': '20', 'DSR ID': '163.1.249.162', 'Date': '01/07/15', 'DisplayColor': '0', 'Exposure Scale': '0.000001', 'Exposure Time': '109', 'Filename': '5691', 'Focus Offset': '-0.000500', 'ICC Profile': 'ScanScope v1', 'ImageID': '5691', 'Left': '18.039850', 'LineAreaXOffset': '0.002305', 'LineAreaYOffset': '-0.000948', 'LineCameraSkew': '-0.001426', 'MPP': '0.5019', 'OriginalHeight': '46415', 'OriginalWidth': '46736', 'ScanScope ID': 'SS1774', 'StripeWidth': '2032', 'Time': '20:47:36', 'Time Zone': 'GMT+00:00', 'Top': '23.975338', 'User': '309d004c-f533-4c56-9b00-12

100%|██████████| 4/4 [00:06<00:00,  1.63s/it]


**** Processing ****  17_010_CG_aSyn_x200b.svs
Error loading Vips Image 17_010_CG_aSyn_x200b.svs
**** Processing ****  PD088_Syn1_CG.svs
**** Processing ****  14_075_CG_aSyn_x200.svs
Error loading Vips Image 14_075_CG_aSyn_x200.svs
**** Processing ****  12_060_CG_aSyn_x200.svs
Error loading Vips Image 12_060_CG_aSyn_x200.svs
**** Processing ****  01_156_Syn1_CG_200x.svs
{'': 'label, macro, thumbnail', 'AppMag': '20', 'DSR ID': '163.1.249.162', 'Date': '01/07/15', 'DisplayColor': '0', 'Exposure Scale': '0.000001', 'Exposure Time': '109', 'Filename': '5694', 'Focus Offset': '-0.000500', 'ICC Profile': 'ScanScope v1', 'ImageID': '5694', 'Left': '15.875823', 'LineAreaXOffset': '0.002305', 'LineAreaYOffset': '-0.000948', 'LineCameraSkew': '-0.001426', 'MPP': '0.5019', 'OriginalHeight': '31575', 'OriginalWidth': '56896', 'ScanScope ID': 'SS1774', 'StripeWidth': '2032', 'Time': '20:58:55', 'Time Zone': 'GMT+00:00', 'Top': '21.002644', 'User': '309d004c-f533-4c56-9b00-125ea9a337c1', 'comment':

...

**** Processing ****  11_063_CG_aSyn_x200.svs
Error loading Vips Image 11_063_CG_aSyn_x200.svs
**** Processing ****  PD061_Syn1_CG.svs
**** Processing ****  PD149_Syn1_CG.svs
**** Processing ****  14_036_CG_aSyn_x200.svs
Error loading Vips Image 14_036_CG_aSyn_x200.svs
**** Processing ****  PD013_Syn1_CG.svs
**** Processing ****  PD001_Syn1_CG.svs
**** Processing ****  02_021_Syn1_CG_200x.svs
{'': 'label, macro, thumbnail', 'AppMag': '20', 'DSR ID': '163.1.249.162', 'Date': '01/07/15', 'DisplayColor': '0', 'Exposure Scale': '0.000001', 'Exposure Time': '109', 'Filename': '5700', 'Focus Offset': '-0.000500', 'ICC Profile': 'ScanScope v1', 'ImageID': '5700', 'Left': '14.203670', 'LineAreaXOffset': '0.002305', 'LineAreaYOffset': '-0.000948', 'LineCameraSkew': '-0.001426', 'MPP': '0.5019', 'OriginalHeight': '36298', 'OriginalWidth': '48768', 'ScanScope ID': 'SS1774', 'StripeWidth': '2032', 'Time': '21:20:21', 'Time Zone': 'GMT+00:00', 'Top': '19.047007', 'User': '309d004c-f533-4c56-9b00-12

  0%|          | 0/5 [00:00<?, ?it/s]

Overlap


100%|██████████| 5/5 [00:52<00:00, 10.56s/it]


**** Processing ****  18_031_CG_aSyn_x200.svs
Error loading Vips Image 18_031_CG_aSyn_x200.svs
**** Processing ****  PD269_Syn1_CG.svs
**** Processing ****  PD207_Syn1_CG.svs
**** Processing ****  PD200_Syn1_CG.svs
**** Processing ****  14_073_CG_aSyn_x200.svs
Error loading Vips Image 14_073_CG_aSyn_x200.svs
**** Processing ****  PD131_Syn1_CG.svs
**** Processing ****  15_007_CG_aSyn_x200.svs
Error loading Vips Image 15_007_CG_aSyn_x200.svs
**** Processing ****  PD017_Syn1_CG.svs


## Train Test Split

In [1]:
DATASET_PATH = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/patient_wise_crops_LB/"
patient_folder = os.listdir(DATASET_PATH)

NameError: name 'os' is not defined

In [53]:
wsi_names = []
count_crops = []
for i in patient_folder:
    wsi_names.append(i)
    count_crops.append(len((glob.glob(os.path.join(DATASET_PATH,i,"images","*")))))
    
wsi_crop_count = pd.DataFrame({"wsi_name":wsi_names,"count_crops":count_crops})

In [54]:
wsi_crop_count["PDD_DLB_tag"] = wsi_crop_count["wsi_name"].apply(lambda l: "PDD" if l.startswith("PD") else "DLB")

In [55]:
wsi_crop_count["PDD_DLB_tag"].value_counts()

DLB    27
PDD    27
Name: PDD_DLB_tag, dtype: int64

In [82]:
random_list = random.sample(range(0, 54), 44)

In [83]:
wsi_crop_count["index"] = wsi_crop_count.index

In [84]:
wsi_crop_count["train_test_flag"]=np.where(wsi_crop_count["index"].isin(random_list),"Train","Val")

In [85]:
wsi_crop_count.groupby(["train_test_flag","PDD_DLB_tag"])["wsi_name"].count()

train_test_flag  PDD_DLB_tag
Train            DLB            22
                 PDD            22
Val              DLB             5
                 PDD             5
Name: wsi_name, dtype: int64

In [86]:
wsi_crop_count

,wsi_name,count_crops,PDD_DLB_tag,index,train_test_flag
0,15_005_CG_aSyn_x200.svs,29,DLB,0,Train
1,PD306_Syn1_CG.svs,92,PDD,1,Val
2,PD061_Syn1_CG.svs,49,PDD,2,Train
3,PD067_Syn1_CG.svs,16,PDD,3,Train
4,PD013_Syn1_CG.svs,0,PDD,4,Train
5,18_031_CG_aSyn_x200.svs,35,DLB,5,Val
6,PD271_Syn1_CG.svs,46,PDD,6,Train
7,14_053_CG_aSyn_x200.svs,26,DLB,7,Train
8,PD133_Syn1_CG.svs,21,PDD,8,Train
9,PD090_Syn1_CG.svs,17,PDD,9,Val


In [87]:
wsi_crop_count[wsi_crop_count["train_test_flag"]=="Train"]["count_crops"].sum()

1137

In [88]:
wsi_crop_count[wsi_crop_count["train_test_flag"]=="Val"]["count_crops"].sum()

186

In [90]:
pat =  wsi_crop_count[wsi_crop_count["train_test_flag"]=="Train"]["wsi_name"].values
TARGET_PATH = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train1'

for i in pat:
   for folder in ["images","labels"]:
      if i==".DS_Store":
         continue
      origin = os.path.join(DATASET_PATH,i,folder)
      target = os.path.join(TARGET_PATH,folder)
      if not os.path.exists(target):
         os.makedirs(target)
      #print(origin)
      # Fetching the list of all the files
      files = os.listdir(origin)
      #print(files)
      # Fetching all the files to directory
      for file_name in files:
         shutil.copyfile(os.path.join(origin,file_name), os.path.join(target,file_name))
      print("Files are copied successfully")

Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are 

In [91]:
pat =  wsi_crop_count[wsi_crop_count["train_test_flag"]=="Val"]["wsi_name"].values
TARGET_PATH = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/val1'

for i in pat:
   for folder in ["images","labels"]:
      if i==".DS_Store":
         continue
      origin = os.path.join(DATASET_PATH,i,folder)
      target = os.path.join(TARGET_PATH,i, folder)
      if not os.path.exists(target):
         os.makedirs(target)
      #print(origin)
      # Fetching the list of all the files
      files = os.listdir(origin)
      #print(files)
      # Fetching all the files to directory
      for file_name in files:
         shutil.copyfile(os.path.join(origin,file_name), os.path.join(target,file_name))
      print("Files are copied successfully")

Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully
Files are copied successfully


In [92]:
#wsi_crop_count.to_csv("WSI_info.csv")
wsi_crop_count.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/WSI_info_v1.csv")

## Data Augmentation

In [2]:
def get_randimages_dataug(total_imgs, image_filenames, label_filenames):
    '''
    This Fn generates random files from the original dataset, which will
    be used to perform data augmentation

    Parameters:
    image_filenames -- list of filenames of images
    label_filenames -- list of filenames of labels
    total_imgs -- how many images needs to be augmented

    Return:
    list of random image and label files for performing data augmentation
    '''
    image_filenames.sort()
    label_filenames.sort()
    random_image_file = []
    random_label_file = []

    for i in range(total_imgs):
        random.seed(i)
        random_image_file.append(random.choice(image_filenames))
        random.seed(i)
        random_label_file.append(random.choice(label_filenames))

    return [random_image_file, random_label_file]

In [6]:
def upsample_dataset(dataset_base_dir, random_img_filenames, rand_label_filenames, variations, transforms, dest_img_folder_name, dest_label_folder_name):
    '''
    This Fn will upsample the images by performing data augmentation

    DataAugmentation includes vertical and horizontal flips

    Parameters:
    random_filenames -- the random filenames of the images for performing data augmentation
    file_type -- if it belongs to images or labels
    variations -- how many variations you need from each image for
    generating your data augmented sample

    '''
    i = 0
    aug_img_files = []
    aug_mask_files = []
    # random.seed(500)

    # Make dir where tha augmented file will reside
    aug_img_dir = os.path.join(dataset_base_dir, dest_img_folder_name)
    if not os.path.exists(aug_img_dir):
            os.makedirs(aug_img_dir)
            print("Augmented Directory '%s' created" %aug_img_dir)
    
    aug_mask_dir = os.path.join(dataset_base_dir, dest_label_folder_name)
    if not os.path.exists(aug_mask_dir):
            os.makedirs(aug_mask_dir)
            print("Augmented Directory '%s' created" %aug_mask_dir)

    print(os.listdir(dataset_base_dir))
    print("\nData Augmentation in Progress ...")
    total_imgs = len(random_img_filenames)
    
    for i in range(total_imgs):
        # load the image
        img = Image.open(random_img_filenames[i]).convert("RGB")
        img = np.array(img)
        mask = Image.open(rand_label_filenames[i]).convert('P')
        mask = np.array(mask)

        for j in range(variations):
                transformed = transforms(image=img, mask=mask)
                transformed_img = transformed["image"]
                transformed_img = Image.fromarray(transformed_img)

                #To rename the file with prefix A_
                filename = os.path.basename(random_img_filenames[i])
                filepath = os.path.dirname(random_img_filenames[i])

                aug_file_name = "A_" + str(i) + "_" + str(j) + "_" + filename
                new_file = os.path.join(dataset_base_dir,dest_img_folder_name,
                                        aug_file_name)
                transformed_img.save(new_file)
                aug_img_files.append(new_file)

                transformed_mask = transformed["mask"]
                transformed_mask = Image.fromarray(transformed_mask)

                #To rename the file with prefix A_
                filename = os.path.basename(rand_label_filenames[i])
                filepath = os.path.dirname(rand_label_filenames[i])
                aug_file_name = "A_" + str(i) + "_" + str(j) + "_" + filename
                new_file = os.path.join(dataset_base_dir, dest_label_folder_name,
                                        aug_file_name)
                transformed_mask.save(new_file)
                aug_mask_files.append(new_file)
    return aug_img_files, aug_mask_files

In [7]:
dataset_base_dir = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train/"
image_input = "images"
label_input = "labels"

# Extracting Image File Names
images_input = os.path.join(dataset_base_dir, image_input)
image_path = os.path.join(images_input, '*.png')
image_filenames = glob.glob(image_path)

# Extracting labels File Names
label_input = os.path.join(dataset_base_dir, label_input)
label_path = os.path.join(label_input, '*.png')
label_filenames = glob.glob(label_path)

assert len(image_filenames) != 0 and len(label_filenames) != 0

In [8]:
transforms  = A.Compose([ A.VerticalFlip(p=0.5),
                            A.HorizontalFlip(p=0.5),
                            A.Blur(blur_limit=1),
                            #A.OpticalDistortion(),
                            #A.HueSaturationValue(),
                            A.RandomRotate90(),
                            #A.RandomBrightnessContrast(p=0.2),
                        ])

In [9]:
aug_value = 2000
rand_image_filenames, rand_label_filenames = get_randimages_dataug(aug_value, image_filenames, label_filenames)

In [35]:
## Code used for all transforms including the commented ones
augmented_image_files, augmented_label_files = upsample_dataset(dataset_base_dir, rand_image_filenames, rand_label_filenames, 3, transforms,  "augmented_images", "augmented_labels")

Augmented Directory '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train/augmented_images' created
Augmented Directory '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train/augmented_labels' created
['labels', '.DS_Store', 'augmented_images', 'augmented_labels', 'images']

Data Augmentation in Progress ...


In [10]:
## Code used for all uncommented transforms
augmented_image_files, augmented_label_files = upsample_dataset(dataset_base_dir, rand_image_filenames, rand_label_filenames, 3, transforms,  "augmented_images1", "augmented_labels1")

Augmented Directory '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train/augmented_images1' created
Augmented Directory '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Train_val_LB/train/augmented_labels1' created
['labels', '.DS_Store', 'augmented_images', 'augmented_labels1', 'augmented_images1', 'augmented_labels', 'images']

Data Augmentation in Progress ...


In [22]:
assert len(augmented_image_files) == len(augmented_label_files)

In [23]:
len(augmented_image_files)

6000

In [24]:
len(augmented_label_files)

6000

In [25]:
os.listdir(dataset_base_dir)

['labels', '.DS_Store', 'images']

## Original Crop-wise Data Labels

In [31]:
filenames = []
labels = []
for geofile in f_list_updated_names:
    filename = geofile.replace(".json",".svs")
    print("**** Processing **** ", filename)
    geofile1 = os.path.join(json_path,geofile)
    with open(geofile1) as f:
        data = json.load(f)
    for ele in tqdm(data):
        if len(ele['label'])==1:
            continue
        if ele['label'][1]=='3+' and ele['label'][0] in ["True","Pre"]:
            for region in ele['region_attributes']:

                # Get tileX and tileY
                tileX = region['tiles'][0]['tileId'][0]
                tileY = region['tiles'][0]['tileId'][1]
                tileWidth = region['tiles'][0]['tileBounds']["WH"][0]
                tileHeight = region['tiles'][0]['tileBounds']["WH"][1]
                # crop the image
                # get the bound-x and bounds-y, offset as Vips crops the empty spaces. Qupath does not
                tileX = (tileX * tileWidth)
                tileY = (tileY * tileHeight)
                coords_x, coords_y = zip(*region['points'])
                coords_x = np.array(coords_x)
                coords_y = np.array(coords_y)
                # Remove overlap annotations
                if len(coords_x[coords_x > x2]) > 0 or len(coords_y[coords_y > y2]) > 0:
                    print('Overlap')
                    continue
                label = ele['label'][0]
                file_name = filename + "_" + str(tileX)+"x" + "_" + str(tileY) + "y" + "_" + "image" + ".png"
                filenames.append(file_name)
                labels.append(label)
                
image_label_df = pd.DataFrame({"image_name":filenames, "ground_truth":labels})
image_label_df.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/image_level_ground_truths.csv")

**** Processing ****  PD130_Syn1_CG.svs


100%|██████████| 7/7 [00:00<00:00, 7212.02it/s]


**** Processing ****  PD079_Syn1_CG.svs


100%|██████████| 11/11 [00:00<00:00, 12970.86it/s]


**** Processing ****  PD092_Syn1_FCX.svs


...

100%|██████████| 8/8 [00:00<00:00, 3656.36it/s]


**** Processing ****  15_007_CG_aSyn_x200.svs


100%|██████████| 7/7 [00:00<00:00, 14007.69it/s]


**** Processing ****  PD017_Syn1_CG.svs


100%|██████████| 14/14 [00:00<00:00, 11788.85it/s]
